# Colab 18 — Baseline: pretrained-embedding (ESM2) vs our SNN

**Question (from supervision).** A learned edit-distance approximator must beat a *simple* solution: **pretrained embeddings + a logistic/ridge readout on Levenshtein**. Deeper, this is the **data-agnosticism / abstraction-vs-memorisation** probe: does a 35M protein LM (trained on millions of sequences for a *biological* objective) recover edit-distance as well as our 150K-param conv encoder trained *zero-shot* on synthetic edit-distance pairs?

**ESM2 = data-dependent foil, not a competitor.** It encodes *biological* similarity, not string/edit similarity. We measure how far each representation's edit-distance tracking survives **off its home modality** (AA -> SS -> 3Di).

**Design: a clean 2x2 {SNN, ESM2} x {free, +head}.**
- **free** -- zero-shot geometry (SNN L2; ESM2 cosine). Neither sees a natural pair.
- **+head** -- frozen embedding + Ridge/Logistic on `|e_a - e_b|`, **component-grouped 5-fold CV** (union-find on the pair graph; no protein shared across folds). Isolates *embedding quality* from *readout supervision*. **ESM2+head = the target-supervised upper-control baseline** the supervisor asked for; **SNN+head = its matched control**. (Ordinary pair-KFold would leak protein identity and inflate the heads -- we use grouped CV.)

**Metrics.** Pearson r (calibrated tracking) and **AUROC is-high** -- a *pairwise high-sim discrimination proxy*, **not** proof of full-pool retrieval (cf. `cross_rep.md` section 6).

**Scope: phase-one AA/SS/3Di only.** Plain English (the 4th supervision axis) and full-pool retrieval hits@k are future work.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR='/content/thesis-edit-distance-nn/sampledata/cath'
CACHE='/content/bench_cache'; os.makedirs(CACHE, exist_ok=True)
for f in ['cath_s20_train70.csv.gz','cath_s20_test30.csv.gz','cath_eval.csv.gz','cath_s20_3di.csv.gz']:
    pth=os.path.join(DATA_DIR,f); print(f'{"OK" if os.path.exists(pth) else "MISSING":<8} {pth}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy transformers matplotlib --quiet

In [ ]:
import time, json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, mean_absolute_error
from rapidfuzz.distance import Levenshtein as RFLev
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED); rng=np.random.default_rng(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',device)

## 2. Config + helpers (mirrors colab17b)

In [ ]:
AA_ALPHABET='ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET='HLS'
CHAR_TO_IDX={c:i for i,c in enumerate(AA_ALPHABET)}; PAD_IDX=20; VOCAB=21
MIN_LEN=50; MAX_LEN=200; N_TRAIN=30000; EPOCHS=30; BS=128; K=16
BAND_LOW=0.30; BAND_HIGH=0.70
AA_SET=set(AA_ALPHABET); SS_SET=set(SS_ALPHABET)
is_aa=lambda s: all(c in AA_SET for c in s); is_ss=lambda s: all(c in SS_SET for c in s)
def norm_lev(a,b):
    L=max(len(a),len(b)); return 1.0 if L==0 else 1.0-RFLev.distance(a,b)/L
def encode_pad(seq):
    idx=[CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx+=[PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx,dtype=torch.long)
def perturb(seq,k,abc):
    s=list(seq); abc=list(abc)
    for _ in range(k):
        if len(s)==0: op='ins'
        elif len(s)>=MAX_LEN: op=rng.choice(['sub','del'])
        else: op=rng.choice(['sub','ins','del'])
        if op=='sub': i=rng.integers(0,len(s)); s[i]=rng.choice([c for c in abc if c!=s[i]])
        elif op=='ins': i=rng.integers(0,len(s)+1); s.insert(i,rng.choice(abc))
        else: i=rng.integers(0,len(s)); del s[i]
    return ''.join(s)
def rand_aa(): L=int(rng.integers(MIN_LEN,MAX_LEN+1)); return ''.join(rng.choice(list(AA_ALPHABET),size=L))
def bin_idx(x): return 0 if x<BAND_LOW else (1 if x<BAND_HIGH else 2)

## 3. Pool + eval + component groups (union-find on the eval-pair graph)

In [ ]:
tr=pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'); te=pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')
p=pd.concat([tr,te],ignore_index=True).drop_duplicates('domain_id')
p=p[p.aa_seq.apply(is_aa)]; p=p[p.ss_seq.apply(is_ss)]; p=p[p.aa_seq.str.len()==p.ss_seq.str.len()]
p['L']=p.aa_seq.str.len(); RESCUED={'4z0mC02','3qkaE02'}; inr=p.L.between(MIN_LEN,MAX_LEN)
p=p[inr|p.domain_id.isin(RESCUED)].reset_index(drop=True)
id_to_aa=dict(zip(p.domain_id,p.aa_seq)); id_to_ss=dict(zip(p.domain_id,p.ss_seq))
d3=pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz'); id_to_3di=dict(zip(d3.domain_id,d3['3di'].astype(str)))
ev=pd.read_csv(f'{DATA_DIR}/cath_eval.csv.gz')
ev['3di_score']=[norm_lev(id_to_3di[a],id_to_3di[b]) if (a in id_to_3di and b in id_to_3di) else np.nan
                 for a,b in zip(ev.domain_a,ev.domain_b)]
prots=sorted(set(ev.domain_a)|set(ev.domain_b)); pidx={d:i for i,d in enumerate(prots)}
LOOK={'AA':id_to_aa,'SS':id_to_ss,'3Di':id_to_3di}; SCORE={'AA':'aa_score','SS':'ss_score','3Di':'3di_score'}
# union-find: proteins co-occurring in a pair share a component -> grouped CV cannot leak identity
par=list(range(len(prots)))
def find(x):
    while par[x]!=x: par[x]=par[par[x]]; x=par[x]
    return x
for a,b in zip(ev.domain_a,ev.domain_b):
    ra,rb=find(pidx[a]),find(pidx[b]); par[ra]=rb
pair_group=np.array([find(pidx[a]) for a in ev.domain_a])
print(f'pool={len(p)}  eval_pairs={len(ev)}  proteins={len(prots)}  components={len(set(pair_group))}')

## 4. ESM2 embeddings (frozen, mean-pooled). 35M reproduces the committed numbers; swap a larger variant on GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModel
ESM2_MODEL='facebook/esm2_t12_35M_UR50D'   # e.g. 'facebook/esm2_t33_650M_UR50D' on GPU
tok=AutoTokenizer.from_pretrained(ESM2_MODEL); esm=AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
esm_params=sum(pp.numel() for pp in esm.parameters())

@torch.no_grad()
def esm_embed(seqs, bs=32):
    order=np.argsort([len(s) for s in seqs]); out=[None]*len(seqs)
    for i in range(0,len(order),bs):
        idx=order[i:i+bs]; batch=[seqs[j] for j in idx]
        enc=tok(batch,return_tensors='pt',padding=True,add_special_tokens=True).to(device)
        h=esm(**enc).last_hidden_state
        mask=enc['attention_mask'].clone(); mask[:,0]=0
        for r,l in enumerate(enc['attention_mask'].sum(1)): mask[r,l-1]=0   # drop cls+eos
        m=mask.unsqueeze(-1).float(); e=(h*m).sum(1)/m.sum(1).clamp(min=1)
        e=F.normalize(e,dim=1).cpu().numpy()
        for kk,j in enumerate(idx): out[j]=e[kk]
    return np.stack(out)

esm_emb={}
for mod in ['AA','SS','3Di']:
    cf=f'{CACHE}/esm2_{mod}.npy'
    esm_emb[mod]=np.load(cf) if os.path.exists(cf) else esm_embed([LOOK[mod][d] for d in prots])
    np.save(cf,esm_emb[mod]); print(mod, esm_emb[mod].shape)

## 5. Our SNN (zero-shot, synthetic-AA only) -- same recipe as colab16b/17b

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb=nn.Embedding(VOCAB,32,padding_idx=PAD_IDX)
        s.c1=nn.Conv1d(32,32,3,padding=1); s.c2=nn.Conv1d(32,64,3,padding=1)
        s.pool=nn.AdaptiveAvgPool1d(K); s.fc=nn.Linear(64*K,128)
    def forward(s,x):
        m=(x!=PAD_IDX).float(); e=s.emb(x).permute(0,2,1)
        h=F.relu(s.c1(e)); h=F.relu(s.c2(h)); h=h*m.unsqueeze(1)
        h=s.pool(h).flatten(1); return F.normalize(s.fc(h),p=2,dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder=Enc()
        s.head=nn.Sequential(nn.Linear(128,64),nn.LeakyReLU(0.01),nn.Linear(64,3))
    def forward(s,a,b): return s.head(torch.abs(s.encoder(a)-s.encoder(b)))
class DS(Dataset):
    def __init__(s,pp): s.p=pp
    def __len__(s): return len(s.p)
    def __getitem__(s,i): a,b,l=s.p[i]; return encode_pad(a),encode_pad(b),bin_idx(l)

In [ ]:
print('generating 30k synthetic AA pairs...')
pairs=[]
while len(pairs)<N_TRAIN:
    sd=rand_aa(); L=len(sd); t=float(rng.uniform(0,1)); k=max(0,int(round((1-t)*L)))
    o=perturb(sd,k,AA_ALPHABET)
    if 1<=len(o)<=MAX_LEN: pairs.append((sd,o,norm_lev(sd,o)))
dl=DataLoader(DS(pairs),batch_size=BS,shuffle=True)
model=Clf().to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3); crit=nn.CrossEntropyLoss()
snn_params=sum(pp.numel() for pp in model.parameters()); model.train()
for ep in range(1,EPOCHS+1):
    tot=nb=0
    for a,b,y in dl:
        a,b,y=a.to(device),b.to(device),y.to(device)
        loss=crit(model(a,b),y); opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item(); nb+=1
    if ep%5==0 or ep==1: print(f'  epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
print(f'SNN params={snn_params}  ESM2 params={esm_params}  ratio={esm_params/snn_params:.0f}x')

In [ ]:
@torch.no_grad()
def snn_embed(look):
    model.eval(); out=np.zeros((len(prots),128),dtype=np.float32)
    for i in range(0,len(prots),256):
        ch=prots[i:i+256]; x=torch.stack([encode_pad(look[d]) for d in ch]).to(device)
        out[i:i+len(ch)]=model.encoder(x).cpu().numpy()
    return out
snn_emb={mod:snn_embed(LOOK[mod]) for mod in ['AA','SS','3Di']}

## 6. Metrics: 2x2 {SNN, ESM2} x {free, +head}, component-grouped CV

In [ ]:
def free_metrics(E,a,b,y,hi,is_snn):
    psim=(1.0-np.linalg.norm(E[a]-E[b],axis=1)/2.0) if is_snn else np.sum(E[a]*E[b],axis=1)
    return dict(pearson=float(np.corrcoef(psim,y)[0,1]),spearman=float(spearmanr(psim,y).correlation),
                auroc=float(roc_auc_score(hi,psim)) if 0<hi.sum()<len(hi) else float('nan'))
def head_metrics(E,a,b,y,hi,groups):
    feat=np.abs(E[a]-E[b]); gkf=GroupKFold(5); pred=np.zeros(len(y)); pred_hi=np.zeros(len(y))
    for trn,tst in gkf.split(feat,groups=groups):
        pred[tst]=Ridge(alpha=1.0).fit(feat[trn],y[trn]).predict(feat[tst])
        if 0<hi[trn].sum()<len(trn):
            pred_hi[tst]=LogisticRegression(max_iter=1000,class_weight='balanced').fit(feat[trn],hi[trn]).predict_proba(feat[tst])[:,1]
    return dict(pearson=float(np.corrcoef(pred,y)[0,1]),mae=float(mean_absolute_error(y,pred)),
                auroc=float(roc_auc_score(hi,pred_hi)) if 0<hi.sum()<len(hi) else float('nan'))

res={'snn_params':int(snn_params),'esm_params':int(esm_params),'ratio':float(esm_params/snn_params)}
for mod in ['AA','SS','3Di']:
    sc=SCORE[mod]; m=ev[sc].notna().values; sub=ev[m]
    a=sub.domain_a.map(pidx).values; b=sub.domain_b.map(pidx).values; y=sub[sc].values
    hi=(y>=BAND_HIGH).astype(int); g=pair_group[m]
    res[mod]=dict(n=int(len(y)),n_high=int(hi.sum()),
        snn_free=free_metrics(snn_emb[mod],a,b,y,hi,True), snn_head=head_metrics(snn_emb[mod],a,b,y,hi,g),
        esm_free=free_metrics(esm_emb[mod],a,b,y,hi,False), esm_head=head_metrics(esm_emb[mod],a,b,y,hi,g))
json.dump(res,open(f'{CACHE}/colab18_results.json','w'),indent=2)

print(f'PEARSON r{"":<6}{"SNN free":>10}{"SNN+head":>10}{"ESM2 free":>10}{"ESM2+head":>10}')
for mod in ['AA','SS','3Di']:
    r=res[mod]; print(f'{mod:<8}n_hi={r["n_high"]:<4}{r["snn_free"]["pearson"]:>10.3f}{r["snn_head"]["pearson"]:>10.3f}{r["esm_free"]["pearson"]:>10.3f}{r["esm_head"]["pearson"]:>10.3f}')
print(f'\nAUROC is-high{"":<2}{"SNN free":>10}{"SNN+head":>10}{"ESM2 free":>10}{"ESM2+head":>10}')
for mod in ['AA','SS','3Di']:
    r=res[mod]; print(f'{mod:<14}{r["snn_free"]["auroc"]:>10.3f}{r["snn_head"]["auroc"]:>10.3f}{r["esm_free"]["auroc"]:>10.3f}{r["esm_head"]["auroc"]:>10.3f}')
print(f'\nparams: SNN={snn_params}  ESM2={esm_params}  ratio={esm_params/snn_params:.0f}x')

In [ ]:
import matplotlib.pyplot as plt
mods=['AA','SS','3Di']; x=np.arange(len(mods)); w=0.2
def col(metric,key):
    return [res[m][key][metric] for m in mods]
fig,ax=plt.subplots(1,2,figsize=(13,4))
for ax_i,met,ttl,lo in [(0,'pearson','Pearson r vs true edit-distance',0),(1,'auroc','AUROC is-high (pairwise proxy)',0.8)]:
    ax[ax_i].bar(x-1.5*w,col(met,'snn_free'),w,label='SNN free')
    ax[ax_i].bar(x-0.5*w,col(met,'snn_head'),w,label='SNN+head')
    ax[ax_i].bar(x+0.5*w,col(met,'esm_free'),w,label='ESM2 free')
    ax[ax_i].bar(x+1.5*w,col(met,'esm_head'),w,label='ESM2+head')
    ax[ax_i].set_title(ttl); ax[ax_i].set_xticks(x); ax[ax_i].set_xticklabels(mods); ax[ax_i].set_ylim(lo,1.0); ax[ax_i].legend(fontsize=8)
plt.tight_layout(); plt.savefig('colab18_baseline.png',dpi=120); plt.show()

## 7. Reading (leak-free, grouped CV)

1. **Zero-shot (free vs free): the SNN beats ESM2 on every metric, every modality** -- widest at 3Di Pearson (~0.64 vs 0.31). No CV, no leakage -> the robust headline. The SNN's untrained-on-target geometry tracks edit-distance better than a generic protein LM's. **Data-agnosticism premium, measured.**
2. **Matched readout (both +head, grouped CV) -- the embedding-quality test.** On **discrimination (AUROC)** the SNN embedding wins everywhere (SS ~0.95 vs ~0.88; 3Di ~0.98 vs ~0.95; AA tie). On **calibrated Pearson** the SNN wins SS but **ESM2 edges 3Di** (~0.69 vs ~0.65). At equal supervision: SNN is the better *ranking* substrate; ESM2 is marginally better for *calibrated 3Di regression*.
3. **"Usable but not directly aligned," made precise.** ESM2's 3Di edit-signal is *latent* (free r~0.31 -> head ~0.69, +0.38); the SNN's is *already aligned* (free ~0.64 ~ head ~0.65). Pretrained biology *contains* edit-distance info but needs a readout to surface it; the SNN exposes it directly in distance.
4. **ESM2 is a *strong* baseline, not broken.** No SS collapse (free r~0.75) -- low-entropy edit-distance ~ length+composition, captured even reading "HLS" as nonsense. The free contrast sharpens on higher-entropy 3Di.

**Caveats.** AA Pearson is uninformative (n_high=5, narrow label range; both heads collapse under grouped CV -- read AA on AUROC only, where all ~tie & underpowered). AUROC is a *pairwise discrimination proxy, not full-pool retrieval* -- hits@k is future work and required before deployment claims. Scope = phase-one AA/SS/3Di; English is the next axis. ESM2-35M is conservative (bigger lifts AA, stays generic off-modality).

**Net (thesis-safe).** Zero-shot SNN geometry beats raw ESM2 geometry across AA/SS/3Di; at matched supervised readout the SNN embedding still wins discrimination everywhere, ESM2 edges only calibrated 3Di regression -- at ~230x fewer parameters and zero natural-pair supervision. A genuinely useful baseline; not a clean sweep, but the leak-free story is the stronger one.